# Silver → Gold: fact_sales

**Propósito:** Construir la tabla de hechos limpia para el modelo semántico a partir de `dbo.salestrack_sales_final` en Silver.

**Transformaciones aplicadas:**
- Drop de columnas desnormalizadas (ya disponibles en las dimensiones Gold):
  - `customer_name` → disponible en `dim_customer.name`
  - `delegate_name` → disponible en `dim_delegate.delegate_name`
  - `province_name` → disponible en `dim_customer.region`
- Validación de integridad referencial contra `dim_customer` y `dim_calendar` en Gold
- Columna de auditoría `_gold_load_ts`

**Star schema resultante:**
```
fact_sales
  ├── date          → dim_calendar.date
  ├── customer_id   → dim_customer.customer_id
  └── delegate_id   → dim_delegate.delegate_id
```

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [ ]:
%run ./config

SILVER_TABLE       = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE         = f"{DEFAULT_SCHEMA}.fact_sales"

# Dimensiones Gold para validación referencial
GOLD_DIM_CUSTOMER  = f"{DEFAULT_SCHEMA}.dim_customer"
GOLD_DIM_CALENDAR  = f"{DEFAULT_SCHEMA}.dim_calendar"
GOLD_DIM_DELEGATE  = f"{DEFAULT_SCHEMA}.dim_delegate"

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas leídas desde Silver: {df_silver.count()}")
df_silver.printSchema()

In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
COLS_TO_DROP = [
    "customer_name",   # en dim_customer.name
    "delegate_name",   # en dim_delegate.delegate_name
    "province_name",   # en dim_customer.region
    "_silver_load_ts",
]

existing_cols = df_silver.columns
cols_to_drop  = [c for c in COLS_TO_DROP if c in existing_cols]

df_base = (
    df_silver
    .drop(*cols_to_drop)
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

# ─── FILTRO REFERENCIAL (evita BLANKs en Power BI) ────────────────────────────
# Inner join contra cada dim: elimina filas cuya FK no existe en la dimensión
df_dim_customer = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CUSTOMER}")
df_dim_calendar = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_CALENDAR}")
df_dim_delegate = spark.read.table(f"`{GOLD_LAKEHOUSE}`.{GOLD_DIM_DELEGATE}")

rows_before = df_base.count()

df_gold = (
    df_base
    .join(df_dim_customer.select("customer_id"), on="customer_id", how="inner")
    .join(df_dim_calendar.select("date"),        on="date",        how="inner")
    .join(df_dim_delegate.select("delegate_id"), on="delegate_id", how="inner")
)

rows_after = df_gold.count()
print(f"Filas antes del filtro referencial : {rows_before}")
print(f"Filas después del filtro           : {rows_after}")
print(f"Filas descartadas (huérfanas)      : {rows_before - rows_after}")

In [ ]:
# ─── VALIDACIÓN DE INTEGRIDAD REFERENCIAL ─────────────────────────────────────
# Las dims ya están cargadas en la celda anterior; reutilizamos las variables.
# Tras el inner join, los huérfanos deben ser siempre 0.

def check_referential_integrity(df_fact, fk_col, df_dim, pk_col, label):
    orphans = (
        df_fact.select(fk_col)
        .join(df_dim.select(pk_col), df_fact[fk_col] == df_dim[pk_col], "left_anti")
        .count()
    )
    status = "✓ OK" if orphans == 0 else f"⚠ {orphans} huérfanos"
    print(f"  Integridad {label}: {status}")
    return orphans

print("Validando integridad referencial...")
orphan_total = 0
orphan_total += check_referential_integrity(df_gold, "customer_id", df_dim_customer, "customer_id", "customer_id → dim_customer")
orphan_total += check_referential_integrity(df_gold, "date",        df_dim_calendar, "date",        "date → dim_calendar")
orphan_total += check_referential_integrity(df_gold, "delegate_id", df_dim_delegate, "delegate_id", "delegate_id → dim_delegate")

In [ ]:
# ─── VALIDACIÓN GENERAL ───────────────────────────────────────────────────────
row_count       = df_gold.count()
null_dates      = df_gold.filter(F.col("date").isNull()).count()
null_customers  = df_gold.filter(F.col("customer_id").isNull()).count()
anomaly_count   = df_gold.filter(F.col("dq_flag_unidades_anomaly") == True).count()

print(f"Filas a escribir en Gold           : {row_count}")
print(f"Filas con date nulo                : {null_dates}")
print(f"Filas con customer_id nulo         : {null_customers}")
print(f"Filas con unidades anómalas (flag) : {anomaly_count}")

assert null_dates     == 0, "ERROR: hay fechas nulas en fact_sales Gold"
assert null_customers == 0, "ERROR: hay customer_id nulos en fact_sales Gold"

df_gold.show(3)

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"✓ Tabla {GOLD_LAKEHOUSE}.{GOLD_TABLE} escrita correctamente con {row_count} filas.")